# LoRA Research Notes Adapter — Demo

This notebook walks through the full pipeline: load the best adapter, run inference on a custom prompt, and evaluate the output against the format template.

**Requirements:** run from the repo root with the project's virtual environment active.
```bash
source .venv/bin/activate
jupyter notebook notebooks/demo.ipynb
```

## 1 · Setup

In [ ]:
import sys, json, textwrap
from pathlib import Path

ROOT = Path().resolve().parent  # repo root when notebook is in notebooks/
sys.path.insert(0, str(ROOT / 'src'))

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

from eval_template import build_prompt, check_compliance, detect_device
from eval_rubric   import rubric_score
from constants     import SYSTEM_PROMPT

print('torch:', torch.__version__)
device = detect_device()
print('device:', device)

## 2 · Pick an adapter

By default we use the best experiment (`rank_64`). Change `ADAPTER_NAME` to any experiment in `outputs/experiments/`.

In [ ]:
ADAPTER_NAME = 'rank_64'   # change to: baseline / rank_8 / rank_32 / epochs_5 / lr_1e-4 / lr_5e-4
ADAPTER_DIR  = ROOT / 'outputs' / 'experiments' / ADAPTER_NAME

meta = json.loads((ADAPTER_DIR / 'training_meta.json').read_text())
MODEL_ID = meta['model_id']
print(f'Base model : {MODEL_ID}')
print(f'Adapter    : {ADAPTER_NAME}  (r={meta["lora_r"]}, lr={meta["learning_rate"]}, epochs={meta["epochs"]})')

## 3 · Load base model + adapter

In [ ]:
print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Loading base model...')
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
    device_map={'': device},
    trust_remote_code=True,
)
base.eval()

print('Attaching LoRA adapter...')
model = PeftModel.from_pretrained(base, str(ADAPTER_DIR))
model.eval()
print('Ready.')

## 4 · Inference helper

In [ ]:
def generate(prompt_text: str, max_new_tokens: int = 256) -> str:
    prompt = build_prompt(prompt_text, tokenizer)
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

print('generate() ready')

## 5 · Try your own prompt

In [ ]:
MY_PROMPT = """
Mixture of Experts (MoE) is an architecture that activates only a sparse subset
of specialist sub-networks (experts) for each input token, allowing very large
total parameter counts while keeping per-token computation constant.
Models such as Mixtral 8x7B use a learned routing mechanism to select the
top-k experts per token at each layer.
""".strip()

output = generate(MY_PROMPT)
print(output)

## 6 · Evaluate the output

In [ ]:
compliance = check_compliance(output)
rubric     = rubric_score(output)

print('=== Format compliance ===')
for k, v in compliance.items():
    mark = '✓' if v else '✗'
    print(f'  {mark}  {k}')

print(f'\n=== Rubric score: {rubric["total"]} / 12 ===')
for dim in ('summary', 'bullets', 'limitation', 'followup'):
    print(f'  {dim:<12} {rubric[dim]} / 3')

## 7 · Side-by-side: base vs LoRA on the held-out test set

Run this cell to see the 10 held-out prompts evaluated against both the raw base model and the LoRA adapter.

In [ ]:
TEST_PATH = ROOT / 'data' / 'test_prompts.jsonl'
prompts   = [json.loads(l) for l in TEST_PATH.read_text().splitlines() if l.strip()]

rows = []
for p in prompts:
    # Base (adapter disabled)
    model.disable_adapter_layers()
    base_out   = generate(p['input'])
    base_comp  = check_compliance(base_out)
    base_rub   = rubric_score(base_out)

    # LoRA adapter
    model.enable_adapter_layers()
    lora_out   = generate(p['input'])
    lora_comp  = check_compliance(lora_out)
    lora_rub   = rubric_score(lora_out)

    rows.append(dict(
        id=p['id'],
        base_pass=base_comp['all_pass'], base_rubric=base_rub['total'],
        lora_pass=lora_comp['all_pass'], lora_rubric=lora_rub['total'],
    ))
    status = '✓' if lora_comp['all_pass'] else '✗'
    print(f"{p['id']}  base={base_rub['total']:4.1f}/12  lora={lora_rub['total']:4.1f}/12  {status}")

base_rate = sum(r['base_pass'] for r in rows) / len(rows)
lora_rate = sum(r['lora_pass'] for r in rows) / len(rows)
print(f"\nCompliance  base={base_rate:.0%}  lora={lora_rate:.0%}")
print(f"Avg rubric  base={sum(r['base_rubric'] for r in rows)/len(rows):.2f}/12  "
      f"lora={sum(r['lora_rubric'] for r in rows)/len(rows):.2f}/12")